In [1]:
pip install pandas openpyxl streamlit scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 39.2 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()

Saving movies_cleaned.xlsx to movies_cleaned.xlsx


In [5]:
import pandas as pd
df = pd.read_excel("movies_cleaned.xlsx")
df.head()

,Year,Movie Title,Genre,Runtime (min),IMDb Rating,Top Figure
0,2000,Gladiator,"Action, Drama",155,8.5,Russell Crowe
1,2000,Memento,"Mystery, Thriller",113,8.4,Guy Pearce
2,2000,X‑Men,"Action, Sci‑Fi",104,7.4,Hugh Jackman
3,2000,Cast Away,"Adventure, Drama",143,7.8,Tom Hanks
4,2000,Chicken Run,"Animation, Comedy",84,7.1,Mel Gibson


In [6]:
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           260 non-null    int64 
 1   Movie Title    260 non-null    object
 2   Genre          260 non-null    object
 3   Runtime (min)  260 non-null    int64 
 4   IMDb Rating    260 non-null    object
 5   Top Figure     260 non-null    object
dtypes: int64(2), object(4)
memory usage: 12.3+ KB


,0
Year,0
Movie Title,0
Genre,0
Runtime (min),0
IMDb Rating,0
Top Figure,0


In [7]:
def recommend_movies(df, genre=None, actor=None, year=None, min_rating=None, top_n=5):
    result = df.copy()

    if genre:
        result = result[result["Genre"].str.contains(genre, case=False, na=False)]

    if actor:
        result = result[result["Top Figure"].str.contains(actor, case=False, na=False)]

    if year:
        result = result[result["Year"] == year]

    if min_rating:
        result = result[result["IMDb Rating"] >= min_rating]

    result = result.sort_values(by=["IMDb Rating", "Year"], ascending=[False, False])

    return result[["Year", "Movie Title", "Genre", "Runtime (min)", "IMDb Rating", "Top Figure"]].head(top_n)

In [8]:
recommend_movies(df, genre="Action")

,Year,Movie Title,Genre,Runtime (min),IMDb Rating,Top Figure
250,2025,Avatar: Fire and Ash,"Action, Adventure",180,—,Sam Worthington
252,2025,Jurassic World: Rebirth,"Action, Adventure",150,—,Chris Pratt
253,2025,Superman,"Action, Adventure",140,—,David Corenswet
258,2025,Predator: Badlands,"Action, Sci‑Fi",118,—,Boyd Holbrook
259,2025,The Fantastic Four: First Steps,"Action, Adventure",135,—,Pedro Pascal


In [9]:
recommend_movies(df, actor="Tom Cruise")

,Year,Movie Title,Genre,Runtime (min),IMDb Rating,Top Figure
221,2022,Top Gun: Maverick,"Action, Drama",130,8.2,Tom Cruise
149,2014,Edge of Tomorrow,"Action, Sci‑Fi",113,7.9,Tom Cruise
185,2018,Mission: Impossible – Fallout,"Action, Adventure",147,7.7,Tom Cruise
24,2002,Minority Report,"Action, Sci‑Fi",145,7.6,Tom Cruise
46,2004,Collateral,"Crime, Thriller",120,7.5,Tom Cruise


In [11]:
processed_df = df.copy()
processed_df['IMDb Rating'] = pd.to_numeric(processed_df['IMDb Rating'], errors='coerce')
processed_df = processed_df.dropna(subset=['IMDb Rating'])

recommend_movies(processed_df, genre="Drama", min_rating=8)

,Year,Movie Title,Genre,Runtime (min),IMDb Rating,Top Figure
190,2019,Parasite,"Comedy, Drama",132,8.6,Song Kang‑ho
140,2014,Interstellar,"Adventure, Drama",169,8.6,Matthew McConaughey
21,2002,City of God,"Crime, Drama",130,8.6,Alexandre Rodrigues
230,2023,Oppenheimer,"Biography, Drama",180,8.5,Cillian Murphy
141,2014,Whiplash,"Drama, Music",106,8.5,Miles Teller


In [12]:
import re

KNOWN_GENRES = [
    "Action", "Adventure", "Animation", "Biography", "Comedy", "Crime",
    "Drama", "Fantasy", "Horror", "Music", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Sport", "Thriller", "War", "Western"
]

def parse_query(text):
    text_lower = text.lower()

    genre = None
    for g in KNOWN_GENRES:
        if g.lower() in text_lower:
            genre = g
            break

    year_match = re.search(r"\b(2000|2001|2002|2003|2004|2005|2006|2007|2008|2009|2010|2011|2012|2013|2014|2015|2016|2017|2018|2019|2020|2021|2022|2023|2024|2025)\b", text)
    year = int(year_match.group()) if year_match else None

    rating_match = re.search(r"(\d+(\.\d+)?)", text_lower)
    min_rating = float(rating_match.group(1)) if rating_match else None

    actor = None
    return genre, actor, year, min_rating

In [ ]:
def chatbot():
    print("Movie Bot ready. Type 'exit' to stop.\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            break

        genre, actor, year, min_rating = parse_query(user_input)
        recs = recommend_movies(df, genre=genre, actor=actor, year=year, min_rating=min_rating)

        if recs.empty:
            print("Bot: No movies found.\n")
        else:
            print("Bot: Here are your recommendations:")
            print(recs.to_string(index=False))
            print()

chatbot()

Movie Bot ready. Type 'exit' to stop.

You: tom 
Bot: Here are your recommendations:
 Year             Movie Title                Genre  Runtime (min) IMDb Rating       Top Figure
 2025    Avatar: Fire and Ash    Action, Adventure            180           —  Sam Worthington
 2025              Zootopia 2 Animation, Adventure            100           — Ginnifer Goodwin
 2025 Jurassic World: Rebirth    Action, Adventure            150           —      Chris Pratt
 2025                Superman    Action, Adventure            140           —  David Corenswet
 2025        Wicked: For Good     Musical, Fantasy            160           —    Cynthia Erivo

You: arnold
Bot: Here are your recommendations:
 Year             Movie Title                Genre  Runtime (min) IMDb Rating       Top Figure
 2025    Avatar: Fire and Ash    Action, Adventure            180           —  Sam Worthington
 2025              Zootopia 2 Animation, Adventure            100           — Ginnifer Goodwin
 2025 Juras